# 居住地・可住地人口密度作成手順書

---

## 1. 目的

本手順書は、郵便番号ポリゴン単位で以下の指標を作成するための手順をまとめたものである。

- 可住地人口密度（Population density on habitable land）
- 居住地人口密度（Population density on residential land）

本指標は、土地利用情報と人口メッシュデータを統合することで作成する。

---

## 2. 使用データ

### 2.1 土地利用情報

- データ名：国土数値情報 土地利用細分メッシュデータ（2021年）
- 解像度：100mメッシュ
- URL：https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-L03-b-2021.html

#### 使用する土地利用区分（L03b_002）

| コード | 内容 |
|--|--|
| 0500 | 森林 |
| 0700 | 建物用地 |
| 1100 | 河川地及び湖沼 |
| 1400 | 海浜 |
| 1500 | 海水域 |

---

### 2.2 人口データ

- データ名：500mメッシュ別将来推計人口データ（2020年国勢調査ベース）
- 解像度：500mメッシュ
- URL：https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-mesh500r6.html

#### 使用変数

- `PTN_2020`：2020年人口

---

### 2.3 郵便番号ポリゴン

- データ名：GEOSPACE 郵便番号ポリゴン（2022年）
- 使用項目：
  - 郵便番号
  - 市区町村コード

---

## 3. 指標の定義

### 3.1 面積指標

- 総面積  
  → 郵便番号ポリゴンの面積

- 可住地面積  
  → 総面積 − 森林 − 水域 − 海浜 − 海水域

- 居住地面積  
  → 建物用地（0700）

---

### 3.2 人口密度指標

- 可住地人口密度  
  = 人口 / 可住地面積（km²）

- 居住地人口密度  
  = 人口 / 居住地面積（km²）

---

## 4. 作成手順

### Step 1：土地利用データの統合

1. 都道府県ごとに提供されている土地利用細分メッシュ（100m）をダウンロード
2. 全ファイルを結合して全国データを作成（GeoPackage等）

---

### Step 2：郵便番号ポリゴンとの重ね合わせ

1. 郵便番号ポリゴンを読み込み
2. 土地利用データとオーバーレイ（intersection）
3. 郵便番号単位で土地利用面積を集計：郵便番号 × 土地利用区分 → 面積合計
4. 以下を算出
・森林面積
・建物用地面積
・水域面積
・海浜面積
・海水域面積

### Step 3：可住地・居住地面積の算出
可住地面積 = 総面積 − 森林 − 水域 − 海浜 − 海水域
居住地面積 = 建物用地

### Step 4：人口データの割当
1. 500mメッシュ人口データを読み込み
2. 郵便番号ポリゴンとオーバーレイ
3. 面積按分により人口を割当

人口（郵便番号） = Σ（メッシュ人口 × 重なり面積比）

### Step 5：人口密度の計算
可住地人口密度 = 人口 / (可住地面積 / 1,000,000)
居住地人口密度 = 人口 / (居住地面積 / 1,000,000)

## 5. 技術的留意点
・面積計算はメートル単位で行う

・メッシュ解像度の不一致により外れ値が生じる可能性がある
本分析では
	•	人口：500mメッシュ
	•	土地利用：100mメッシュ
を使用しているため、
	•	小さい居住地に人口が集中して割り当てられる
	•	居住地人口密度が過大になる
という問題が発生する可能性がある

・外れ値の発生
以下の条件で極端値が発生する
	•	居住地面積が極小
	•	非居住地（空港・山地等）に人口が割り当てられる

## 6 対応方法

### 6.1 面積によるフィルタ
    居住地面積が極端に小さい郵便番号を除外
### 6.2 比率によるフィルタ
    居住地割合（居住地 / 総面積）が一定以下の地域を除外
### 6.3 分布の補正
    上位1%のwinsorizeやlog変換を検討
### 6.4 指標の使い分け
    可住地人口密度：安定性が高い
    居住地人口密度：実態に近いが不安定
## 7. 限界
	•	人口はメッシュ単位であり、実際の居住位置とは一致しない
	•	建物用地には住宅以外（商業・工業）が含まれる
	•	郵便番号ポリゴンは居住区単位ではない

## 8 出力データ

CSV
	•	zipcode_landuse_area.csv

GPKG
	•	zipcode_landuse_area.gpkg

含まれる主な変数：
	•	zipcode
	•	total_area_m2
	•	habitable_area_m2
	•	residential_area_m2
	•	population_2020
	•	pop_density_habitable
	•	pop_density_residential

## 9. 今後の改善
	•	250m人口メッシュの導入（精度向上）
	•	住宅地のみ抽出した土地利用分類の検討
	•	町丁目人口データとの比較